In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pandas as pd

load_dotenv(dotenv_path=r"C:\Dev\india-air-quality-intel\.env", override=True)

db_host = os.getenv("MYSQL_HOST")
db_port = os.getenv("MYSQL_PORT")
db_user = os.getenv("MYSQL_USER")
db_password = os.getenv("MYSQL_PASSWORD")
db_name = os.getenv("MYSQL_DATABASE")

print("Connecting as user:", db_user)   # sanity check — should print 'aqi_loader', not 'root'

engine = create_engine(f"mysql+mysqlconnector://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")

df_check = pd.read_sql("SELECT COUNT(*) AS row_count FROM fact_kaggle_historical;", engine)
print(df_check)

Connecting as user: aqi_loader
   row_count
0    1140696


In [2]:
import pandas as pd

query = """
SELECT f.daily_aqi_id, f.station_id, s.station_name, s.source_station_key,
       c.city_name, f.reading_date, f.aqi_value, f.aqi_bucket, f.flag_missing
FROM fact_kaggle_daily_aqi f
JOIN dim_station s ON f.station_id = s.station_id
JOIN dim_city c ON s.city_id = c.city_id
WHERE s.source_system = 'kaggle'
"""
df_raw = pd.read_sql(query, engine)
print(len(df_raw))

95058


In [3]:
# Step A: drop rows with missing AQI values
df_valid = df_raw[df_raw['flag_missing'] == 0].copy()
print("After flag_missing filter:", len(df_valid))

# Step B: exclude Ahmedabad's GJ001 entirely
df_model = df_valid[df_valid['source_station_key'] != 'GJ001'].copy()
print("After excluding GJ001:", len(df_model))

# Sanity checks
print(df_model['city_name'].nunique(), "cities remain")
print(sorted(df_model['city_name'].unique()))
print("Pune rows:", (df_model['city_name'] == 'Pune').sum())
print("Ahmedabad rows:", (df_model['city_name'] == 'Ahmedabad').sum())

After flag_missing filter: 76160
After excluding GJ001: 74826
9 cities remain
['Bengaluru', 'Chennai', 'Delhi', 'Hyderabad', 'Jaipur', 'Kolkata', 'Lucknow', 'Mumbai', 'Patna']
Pune rows: 0
Ahmedabad rows: 0


In [4]:
# Extract calendar month (1-12), independent of year — pools all years together per city/month
df_model['month'] = pd.to_datetime(df_model['reading_date']).dt.month

def mad(series):
    med = series.median()
    return (series - med).abs().median()

baseline = df_model.groupby(['city_name', 'month'])['aqi_value'].agg(
    median_aqi='median',
    n_readings='count'
).reset_index()

mad_vals = df_model.groupby(['city_name', 'month'])['aqi_value'].apply(mad).reset_index(name='mad_aqi')

baseline = baseline.merge(mad_vals, on=['city_name', 'month'])

pd.set_option('display.max_rows', None)
print(baseline.sort_values(['city_name', 'month']))

     city_name  month  median_aqi  n_readings  mad_aqi
0    Bengaluru      1       104.0         885     21.0
1    Bengaluru      2        95.0         806     20.0
2    Bengaluru      3       102.0         848     24.0
3    Bengaluru      4        95.0         789     27.0
4    Bengaluru      5        77.0         831     19.0
5    Bengaluru      6        59.0         818     15.0
6    Bengaluru      7        57.0         744     15.0
7    Bengaluru      8        56.0         734     17.0
8    Bengaluru      9        71.0         754     21.5
9    Bengaluru     10        82.0         745     23.0
10   Bengaluru     11        89.0         821     22.0
11   Bengaluru     12        94.0         864     22.0
12     Chennai      1       104.0         423     36.0
13     Chennai      2        93.0         390     24.0
14     Chennai      3        85.0         421     22.0
15     Chennai      4        79.0         443     25.0
16     Chennai      5        96.0         471     27.0
17     Che

In [6]:
print(baseline[baseline['mad_aqi'] == 0])
print(len(baseline))

Empty DataFrame
Columns: [city_name, month, median_aqi, n_readings, mad_aqi]
Index: []
108


In [7]:
print(baseline.sort_values('n_readings').head(15))

    city_name  month  median_aqi  n_readings  mad_aqi
104     Patna      9       110.0          87     27.0
102     Patna      7        98.0          92     11.5
103     Patna      8       107.0          92     15.0
105     Patna     10       259.0         119     63.0
106     Patna     11       371.0         147     36.0
66    Kolkata      7        58.0         162     14.0
107     Patna     12       386.0         170     37.5
67    Kolkata      8        57.0         185     16.0
97      Patna      2       244.5         200     71.5
56     Jaipur      9        85.0         207     21.0
54     Jaipur      7        88.5         208     21.0
55     Jaipur      8        79.0         211     19.0
100     Patna      5       138.0         229     42.0
68    Kolkata      9        45.0         232     14.5
99      Patna      4       144.0         233     44.0


In [8]:
# Merge each reading against its city/month baseline
df_scored = df_model.merge(baseline, on=['city_name', 'month'], how='left')

# Modified z-score formula, as agreed
df_scored['modified_z'] = 0.6745 * (df_scored['aqi_value'] - df_scored['median_aqi']) / df_scored['mad_aqi']

# Flag rule, as agreed
df_scored['is_anomaly'] = df_scored['modified_z'].abs() > 3.5

print("Total scored readings:", len(df_scored))
print("Total flagged anomalies:", df_scored['is_anomaly'].sum())
print("Anomaly rate: {:.2f}%".format(100 * df_scored['is_anomaly'].mean()))

Total scored readings: 74826
Total flagged anomalies: 2002
Anomaly rate: 2.68%


In [9]:
print(df_scored['modified_z'].isna().sum())

0


In [10]:
print(df_scored[df_scored['is_anomaly']].groupby('city_name').size().sort_values(ascending=False))
print()
print(df_scored[df_scored['is_anomaly']].groupby(['city_name','month']).size().sort_values(ascending=False).head(15))

city_name
Delhi        835
Chennai      377
Bengaluru    316
Hyderabad    167
Lucknow      133
Jaipur        93
Kolkata       44
Mumbai        20
Patna         17
dtype: int64

city_name  month
Delhi      8        167
           7        125
           11       123
           6        122
           9        120
           12        62
Chennai    3         55
Bengaluru  7         53
Chennai    4         49
           7         46
Delhi      1         43
Bengaluru  6         35
Chennai    10        34
Bengaluru  8         32
Chennai    9         30
dtype: int64


In [11]:
print(df_scored.reindex(df_scored['modified_z'].abs().sort_values(ascending=False).index)
      [['city_name','reading_date','aqi_value','median_aqi','mad_aqi','modified_z']].head(15))

       city_name reading_date  aqi_value  median_aqi  mad_aqi  modified_z
59565  Hyderabad   2016-02-22      917.0       111.0     15.0   36.243133
59564  Hyderabad   2016-02-21      779.0       111.0     15.0   30.037733
59664  Hyderabad   2016-06-16      737.0        60.0     18.0   25.368694
71161    Lucknow   2018-08-27      814.0        82.5     21.0   23.495083
59311  Hyderabad   2015-06-02      661.0        60.0     18.0   22.520806
59462  Hyderabad   2015-10-31      896.0        98.0     25.0   21.530040
59690  Hyderabad   2016-07-12      495.0        50.0     14.0   21.439464
59461  Hyderabad   2015-10-30      884.0        98.0     25.0   21.206280
40417  Bengaluru   2019-09-17      727.0        71.0     21.5   20.580093
59523  Hyderabad   2016-01-01      660.0       134.0     19.0   18.673000
64204    Chennai   2016-07-05      636.0        92.0     20.0   18.346400
39487  Bengaluru   2016-07-01      464.0        57.0     15.0   18.301433
39482  Bengaluru   2016-06-24      461

In [12]:
city_totals = df_scored.groupby('city_name').size()
city_flags = df_scored[df_scored['is_anomaly']].groupby('city_name').size()
city_summary = pd.DataFrame({'total_readings': city_totals, 'flagged': city_flags}).fillna(0)
city_summary['flag_rate_pct'] = 100 * city_summary['flagged'] / city_summary['total_readings']
print(city_summary.sort_values('flag_rate_pct', ascending=False))

           total_readings  flagged  flag_rate_pct
city_name                                        
Chennai              4810      377       7.837838
Bengaluru            9639      316       3.278348
Jaipur               3011       93       3.088675
Lucknow              5245      133       2.535748
Hyderabad            7025      167       2.377224
Delhi               36107      835       2.312571
Kolkata              2996       44       1.468625
Patna                2121       17       0.801509
Mumbai               3872       20       0.516529


In [13]:
# Which stations produce the most extreme flagged readings?
top50 = df_scored.reindex(df_scored['modified_z'].abs().sort_values(ascending=False).index).head(50)
print(top50['station_name'].value_counts())
print()

# How many flagged readings exceed the nominal 500 ceiling, and from how many distinct stations?
over_500 = df_scored[(df_scored['is_anomaly']) & (df_scored['aqi_value'] > 500)]
print("Flagged readings over 500:", len(over_500))
print("Distinct stations involved:", over_500['station_name'].nunique())
print(over_500['station_name'].value_counts())

station_name
BWSSB Kadabesanahalli, Bengaluru - CPCB                 25
Sanathnagar, Hyderabad - TSPCB                           8
Anand Vihar, Delhi - DPCC                                7
Peenya, Bengaluru - CPCB                                 3
Alandur Bus Depot, Chennai - CPCB                        2
Shastri Nagar, Jaipur - RSPCB                            2
Talkatora District Industries Center, Lucknow - CPCB     1
Manali, Chennai - CPCB                                   1
Sirifort, Delhi - CPCB                                   1
Name: count, dtype: int64

Flagged readings over 500: 425
Distinct stations involved: 45
station_name
Anand Vihar, Delhi - DPCC                               155
Sirifort, Delhi - CPCB                                   21
Wazirpur, Delhi - DPCC                                   20
DTU, Delhi - CPCB                                        15
Mundka, Delhi - DPCC                                     15
R K Puram, Delhi - DPCC                               

In [14]:
kad = df_scored[df_scored['station_name'] == 'BWSSB Kadabesanahalli, Bengaluru - CPCB']
print(kad['aqi_value'].describe())
print()
print(kad[kad['is_anomaly']][['reading_date','aqi_value','median_aqi','mad_aqi','modified_z']].sort_values('reading_date'))

count    1596.000000
mean       98.977444
std        69.732454
min         8.000000
25%        58.000000
50%        84.000000
75%       114.000000
max       727.000000
Name: aqi_value, dtype: float64

      reading_date  aqi_value  median_aqi  mad_aqi  modified_z
39107   2015-05-02      217.0        77.0     19.0    4.970000
39113   2015-05-08      243.0        77.0     19.0    5.893000
39127   2015-05-22      177.0        77.0     19.0    3.550000
39129   2015-05-24      267.0        77.0     19.0    6.745000
39130   2015-05-25      178.0        77.0     19.0    3.585500
39131   2015-05-26      262.0        77.0     19.0    6.567500
39140   2015-06-05      211.0        59.0     15.0    6.834933
39143   2015-06-08      147.0        59.0     15.0    3.957067
39149   2015-06-14      146.0        59.0     15.0    3.912100
39151   2015-06-16      214.0        59.0     15.0    6.969833
39156   2015-06-21      220.0        59.0     15.0    7.239633
39157   2015-06-22      180.0        59.0  

In [15]:
chennai_flags = df_scored[(df_scored['city_name'] == 'Chennai') & (df_scored['is_anomaly'])]
print(chennai_flags['station_name'].value_counts())
print()
print(chennai_flags.groupby('month').size().sort_values(ascending=False))

station_name
Manali, Chennai - CPCB                 180
Alandur Bus Depot, Chennai - CPCB      125
Velachery Res. Area, Chennai - CPCB     68
Manali Village, Chennai - TNPCB          4
Name: count, dtype: int64

month
3     55
4     49
7     46
10    34
9     30
2     30
5     30
11    25
12    23
6     21
8     18
1     16
dtype: int64


In [16]:
kad_flagged = kad[kad['is_anomaly']].copy()
kad_flagged['year'] = pd.to_datetime(kad_flagged['reading_date']).dt.year
print(kad_flagged.groupby('year').size())
print()
print("Full value range:", kad['aqi_value'].min(), "-", kad['aqi_value'].max())
print("Total readings for this station:", len(kad))

year
2015    49
2016    23
2017    22
2018     2
2019    17
2020     6
dtype: int64

Full value range: 8.0 - 727.0
Total readings for this station: 1596


In [17]:
station_medians = df_model[df_model['city_name'] == 'Chennai'].groupby('station_name')['aqi_value'].median().sort_values(ascending=False)
print(station_medians)

station_name
Manali, Chennai - CPCB                 104.0
Manali Village, Chennai - TNPCB         98.0
Alandur Bus Depot, Chennai - CPCB       94.0
Velachery Res. Area, Chennai - CPCB     78.0
Name: aqi_value, dtype: float64
